In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "revenue-t12-qa"
NOTEBOOK_PATH = "notebooks/qa/42-revenue-t12-qa.ipynb"
TOWER = 12
TOWER_NAME = "Tower of Revenue"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 12: Tower of Revenue


In [2]:
# Cell 1: Pricing Structure in start.html (Floors 1-7)

start_html = read_file(WEB / "start.html")

# P1: start.html mentions Free tier
has_free = "Free" in start_html and ("free" in start_html.lower())
record("P1", "start.html mentions Free tier", has_free,
       "Free tier referenced in landing/start page")

# P2: start.html mentions Starter $8/mo
has_starter = "$8" in start_html and "Starter" in start_html
record("P2", "start.html mentions Starter $8/mo tier", has_starter,
       "Starter tier with $8/mo pricing found")

# P3: start.html mentions Pro $28/mo
has_pro = "$28" in start_html and "Pro" in start_html
record("P3", "start.html mentions Pro $28/mo tier", has_pro,
       "Pro tier with $28/mo pricing found")

# P4: start.html has a primary CTA button
has_cta = "btn-primary" in start_html or "auth-btn--primary" in start_html
record("P4", "start.html has primary CTA button", has_cta,
       "Primary action button class found")

PASS: start.html mentions Free tier -- Free tier referenced in landing/start page
PASS: start.html mentions Starter $8/mo tier -- Starter tier with $8/mo pricing found
PASS: start.html mentions Pro $28/mo tier -- Pro tier with $28/mo pricing found
PASS: start.html has primary CTA button -- Primary action button class found


In [3]:
# Cell 2: Download and Value Pages (Floors 8-14)

# P5: download.html exists with platform options
dl_html = read_file(WEB / "download.html")
has_download_page = len(dl_html) > 100
has_platforms = any(p in dl_html.lower() for p in ["linux", "macos", "windows"])
record("P5", "download.html exists with platform options", has_download_page and has_platforms,
       "Download page with platform-specific options")

# P6: guide.html exists (getting started)
guide_html = read_file(WEB / "guide.html")
has_guide = len(guide_html) > 100
record("P6", "guide.html exists (getting started guide)", has_guide,
       f"Guide page: {len(guide_html)} chars")

# P7: app-store.html exists (skill marketplace)
appstore_html = read_file(WEB / "app-store.html")
has_appstore = len(appstore_html) > 100
record("P7", "app-store.html exists (marketplace)", has_appstore,
       f"App store page: {len(appstore_html)} chars")

PASS: download.html exists with platform options -- Download page with platform-specific options
PASS: guide.html exists (getting started guide) -- Guide page: 13594 chars
PASS: app-store.html exists (marketplace) -- App store page: 11128 chars


In [4]:
# Cell 3: Footer Pricing Link and Navigation (Floors 15-21)

# P8: Footer has pricing link
footer_html = read_file(WEB / "partials-footer.html")
has_pricing_link = "pricing" in footer_html.lower()
record("P8", "Footer has pricing link", has_pricing_link,
       "Pricing link found in partials-footer.html")

# P9: Pricing links to solaceagi.com
links_to_solace = "solaceagi.com" in footer_html
record("P9", "Pricing links to solaceagi.com", links_to_solace,
       "Footer pricing links to hosted platform")

# P10: settings.html exists (subscription management)
settings_html = read_file(WEB / "settings.html")
has_settings = len(settings_html) > 100
record("P10", "settings.html exists (user management)", has_settings,
       f"Settings page: {len(settings_html)} chars")

# P11: BYOK model referenced (bring your own key)
has_byok = "API key" in start_html or "api key" in start_html.lower() or "your own" in start_html.lower()
record("P11", "BYOK (Bring Your Own Key) referenced in start.html", has_byok,
       "Users can bring their own API key")

PASS: Footer has pricing link -- Pricing link found in partials-footer.html
PASS: Pricing links to solaceagi.com -- Footer pricing links to hosted platform
PASS: settings.html exists (user management) -- Settings page: 47147 chars
PASS: BYOK (Bring Your Own Key) referenced in start.html -- Users can bring their own API key


In [5]:
# Cell 4: Support and Trust Pages (Floors 22-28)

# P12: Support/help references exist (footer contact, email, guide, or dedicated page)
support_html = read_file(WEB / "support" / "index.html") if (WEB / "support").is_dir() else read_file(WEB / "support.html")
support_exists = len(support_html) > 0
if not support_exists:
    # Check footer for support/help/contact references (contact link, support email, etc.)
    footer_html = read_file(WEB / "partials-footer.html")
    guide_html_check = read_file(WEB / "guide.html")
    has_support_in_footer = ("support" in footer_html.lower() or
                             "contact" in footer_html.lower() or
                             "help" in footer_html.lower())
    has_guide = len(guide_html_check) > 100
    support_exists = has_support_in_footer or has_guide
    detail = []
    if has_support_in_footer:
        detail.append("footer has support/contact references")
    if has_guide:
        detail.append("guide.html exists as help resource")
    support_detail = "; ".join(detail) if detail else "No support references found"
else:
    support_detail = "Dedicated support page exists"
record("P12", "Support/help resources available to users", support_exists,
       support_detail)

# P13: glossary.html exists (reducing support costs)
glossary_html = read_file(WEB / "glossary.html")
has_glossary = len(glossary_html) > 100
record("P13", "glossary.html exists (self-service help)", has_glossary,
       f"Glossary page: {len(glossary_html)} chars")

# P14: 404.html exists (professional UX)
four04_html = read_file(WEB / "404.html")
has_404 = len(four04_html) > 50
record("P14", "404.html exists (professional error page)", has_404,
       "Custom 404 page for user experience")

# P15: SECURITY.md exists (trust signal)
security_md = (BROWSER_ROOT / "SECURITY.md").exists()
record("P15", "SECURITY.md exists (trust signal)", security_md,
       "Security policy documented for trust")

PASS: Support/help resources available to users -- footer has support/contact references; guide.html exists as help resource
PASS: glossary.html exists (self-service help) -- Glossary page: 8343 chars
PASS: 404.html exists (professional error page) -- Custom 404 page for user experience
PASS: SECURITY.md exists (trust signal) -- Security policy documented for trust


In [6]:
# Cell 5: Value Delivery and Demo (Floors 29-35)

# P16: demo.html exists (first value experience)
demo_html = read_file(WEB / "demo.html")
has_demo = len(demo_html) > 100
record("P16", "demo.html exists (first value experience)", has_demo,
       f"Demo page: {len(demo_html)} chars")

# P17: Pricing tiers mentioned in evidence schedule JS
sched_ev = read_file(WEB / "js" / "schedule-evidence.js")
has_tier_ref = "tier" in sched_ev.lower() or "Starter" in sched_ev or "Pro" in sched_ev
record("P17", "Tier references in schedule-evidence.js", has_tier_ref,
       "Tier-aware features in evidence scheduling")

# P18: FSL license exists (revenue protection)
license_file = read_file(BROWSER_ROOT / "LICENSE")
has_license = len(license_file) > 50
record("P18", "LICENSE file exists (FSL revenue protection)", has_license,
       f"License file: {len(license_file)} chars")

# P19: CONTRIBUTING.md exists (community path)
contributing = (BROWSER_ROOT / "CONTRIBUTING.md").exists()
record("P19", "CONTRIBUTING.md exists (contributor path)", contributing,
       "Community contribution path documented")

PASS: demo.html exists (first value experience) -- Demo page: 31036 chars
PASS: Tier references in schedule-evidence.js -- Tier-aware features in evidence scheduling
PASS: LICENSE file exists (FSL revenue protection) -- License file: 3618 chars
PASS: CONTRIBUTING.md exists (contributor path) -- Community contribution path documented


In [7]:
# Cell 6: Revenue Infrastructure (Floors 36-49)

# P20: start.html has structured data with price
has_schema = '"price"' in start_html or '"priceCurrency"' in start_html
record("P20", "start.html has structured data with pricing", has_schema,
       "Schema.org pricing metadata for SEO")

# P21: Onboarding JS exists (first-run value)
onboarding_js = read_file(WEB / "js" / "onboarding.js")
has_onboarding = len(onboarding_js) > 100
record("P21", "onboarding.js exists (first-run experience)", has_onboarding,
       f"Onboarding script: {len(onboarding_js)} chars")

# P22: Multiple HTML pages exist (complete product surface)
html_count = len(list(WEB.glob("*.html")))
record("P22", "Multiple HTML pages exist (>=15)", html_count >= 15,
       f"Found {html_count} HTML pages in web/")

# P23: Recipe system exists (cost reduction engine)
recipe_dir = SRC / "recipes"
recipe_files = list(recipe_dir.glob("*.py")) if recipe_dir.exists() else []
record("P23", "Recipe system exists (cost reduction engine)", len(recipe_files) >= 3,
       f"Found {len(recipe_files)} recipe module files: {[f.name for f in recipe_files]}")

PASS: start.html has structured data with pricing -- Schema.org pricing metadata for SEO
PASS: onboarding.js exists (first-run experience) -- Onboarding script: 19283 chars
PASS: Multiple HTML pages exist (>=15) -- Found 20 HTML pages in web/
PASS: Recipe system exists (cost reduction engine) -- Found 6 recipe module files: ['recipe_cache.py', 'recipe_executor.py', 'recipe_compiler.py', '__init__.py', 'recipe_parser.py', 'metrics.py']


In [8]:
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 12: Tower of Revenue
Probes: 23 | Passed: 23 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: fc8065e61e227ae9

{
  "surface": "revenue-t12-qa",
  "tower": 12,
  "tower_name": "Tower of Revenue",
  "timestamp": "2026-03-06T11:33:56.924024",
  "total_probes": 23,
  "passed": 23,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "fc8065e61e227ae9"
}
